# **Sorce Files in Drive**

- [filse `.md` In Drive](https://drive.google.com/drive/folders/1rNV2QM_mkd81OEMwvvrhaagn8QNWXAl8?usp=sharing)

---

- [filse after Preprocess `.JSON`](https://drive.google.com/drive/folders/16i5mWolzX10y--c5TJ2kSNiNNfUTtdum?usp=sharing)

---

- [DB in Drive](https://drive.google.com/drive/folders/1pZUszkYY7iiktpfApAUArfchS6qj31nr?usp=sharing)

# **1. Code Extarct Metadata**

In [ ]:
!pip install -q -U \
langchain==0.3.27 \
langchain-core==0.3.79 \
langchain-groq==0.3.8 \
"pydantic<=2.12.3" \
PyYAML

In [ ]:
import pydantic
print(pydantic.__version__)

In [ ]:
import os
os.kill(os.getpid(), 9)

## **Extract MetaData Using LLM**

In [ ]:
import os
import yaml
import time
from typing import Optional

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [ ]:
# =========================================================
# Google Drive
# =========================================================
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive Done")

In [ ]:
# =========================================================
# API
# =========================================================
#os.environ["GROQ_API_KEY"] = "your API"

In [ ]:
# =========================================================
# Schema
# =========================================================
class LegalMetadata(BaseModel):
    document_type: str = Field(..., description="نوع الوثيقة القانونية")
    document_title: str = Field(..., description="الاسم الرسمي للوثيقة")
    issuing_authority: Optional[str] = Field(default=None, description="الجهة المصدرة")
    document_year: Optional[int] = Field(default=None, description="سنة الإصدار")
    decision_number: Optional[str] = Field(default=None, description="رقم القرار أو النظام")
    legal_basis: Optional[str] = Field(default=None, description="السند القانوني")
    effective_date: Optional[str] = Field(default=None, description="تاريخ النفاذ")

In [ ]:
# =========================================================
# Groq
# =========================================================
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=1024,
)

# Structured Output
structured_llm = llm.with_structured_output(LegalMetadata)

# =========================================================
# Prompt
# =========================================================
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
أنت خبير قانوني أردني ومهندس بيانات متخصص في التشريعات الجامعية.

مهمتك:
استخراج Metadata دقيقة من ترويسة الوثائق القانونية.

القواعد:
- أرجع البيانات بصيغة Structured Output فقط.
- إذا لم تجد معلومة، ضع null.
- لا تخمن المعلومات.
- استخرج المعلومات كما هي من النص.
- ركز على الأنظمة والتعليمات الجامعية الأردنية.
"""
    ),
    (
        "human",
        "استخرج البيانات الوصفية من النص التالي:\n\n{header_text}"
    )
])
# LCEL
metadata_chain = prompt | structured_llm
print("Done")

In [ ]:
# =========================================================
# 5. Header Extracion
# =========================================================
def extract_header_text(filepath, lines_to_read=25):
    """
    Read firest Row in the file N
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = []
            for _, line in zip(range(lines_to_read), f):
                lines.append(line)
        return ''.join(lines)
    except Exception as e:
        print(f"❌ Error reading {filepath}: {e}")
        return ""

In [ ]:
# =========================================================
# YAML Frontmatter
# =========================================================
def inject_frontmatter(filepath, metadata_obj):
    metadata_dict = {
        k: v
        for k, v in metadata_obj.model_dump().items()
        if v is not None
    }

    yaml_frontmatter = yaml.dump(
        metadata_dict,
        allow_unicode=True,
        default_flow_style=False,
        sort_keys=False
    )

    frontmatter_block = f"---\n{yaml_frontmatter}---\n\n"

    with open(filepath, 'r', encoding='utf-8') as f:
        original_content = f.read()

    # Prevent Replecation
    if original_content.startswith('---'):
        print(f"Contain Metadata: {filepath}")
        return

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(frontmatter_block + original_content)

In [ ]:
# =========================================================
# Pipeline
# =========================================================
def process_directory_metadata(base_dir):
    print(f"start processing: {base_dir}")

    processed_count = 0
    failed_count = 0

    if not os.path.exists(base_dir):
        print(f"❌ Path not Exist: {base_dir}")
        return

    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if not file.endswith('.md'):
                continue

            filepath = os.path.join(root, file)
            print(f"Processing.....: {file}")

            header_text = extract_header_text(filepath)

            if not header_text.strip():
                print("not found text")
                continue

            try:
                metadata = metadata_chain.invoke({
                    "header_text": header_text
                })

                inject_frontmatter(filepath, metadata)
                processed_count += 1
                print("✅ processing Done")

                # Rate Limit Groq
                time.sleep(2)

            except Exception as e:
                failed_count += 1
                print(f"❌ Error in {file}: {e}")
                # Rate Limit
                time.sleep(5)

    print("\n📊 Final result")
    print(f"✅ success Files: {processed_count}")
    print(f"❌ Fail Files: {failed_count}")

# =========================================================
# Run
# =========================================================
TARGET_DIR = '/content/drive/MyDrive/low_TTU_md'
process_directory_metadata(TARGET_DIR)

# **Prepricess Level 0 `Artical`**
- # **Clean Artical Name**

In [ ]:
import re

def normalize_arabic_numbers(text):
    """
    Converting Eastern Arabic numerals to standard English/Arabic numerals
    To ensure consistency across the entire system.
    """
    eastern_to_western = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
    return text.translate(eastern_to_western)

def standardize_article_headers(text):
    """
    A smart function for discovering and standardizing the names of materials in legal texts.
    """
    # 1. Standardize the numbers throughout the entire text first.
    text = normalize_arabic_numbers(text)

    # 2. معالجة الأخطاء الشائعة الثابتة (مثل المادة و -> المادة 9)
    # نستخدم \b لضمان أن حرف 'و' هو كلمة مستقلة وليس جزءاً من كلمة أخرى
    text = re.sub(r'المادة\s+و\b', 'المادة 9', text)

    lines = text.split('\n')
    normalized_lines = []

    # 3. التعبير النمطي (Regex) الجوهري:
    # ^[\#\-\s]* : تجاهل أي هاشتاجات، شحطات، أو مسافات في البداية
    # المادة\s* : كلمة "المادة" متبوعة بمسافة اختيارية
    # [\(]?\s*(\d+)\s*[\)]? : التقاط الرقم (سواء كان داخل أقواس أم لا) [المجموعة 1]
    # [\:\-\ـ]*\s* : تجاهل أي فواصل عشوائية بعد الرقم (مثل : أو - أو ـ)
    # (.*)$              : التقاط أي نص متبقي في نفس السطر (قد يكون عنواناً للمادة أو بداية النص) [المجموعة 2]
    article_pattern = re.compile(r'^[\#\-\s]*المادة\s*[\(]?\s*(\d+)\s*[\)]?\s*[\:\-\ـ]*\s*(.*)$')

    for line in lines:
        # إزالة هوامش التوثيق العشوائية مثل ² أو ¹ من نهاية السطر
        line = re.sub(r'[¹²³⁴⁵⁶⁷⁸⁹⁰]+$', '', line.strip())

        match = article_pattern.match(line)

        if match:
            article_num = match.group(1)
            remaining_text = match.group(2).strip()

            # تنظيف إضافي للنص المتبقي إذا كان يبدأ بشحطات أو نقطتين مكررة
            remaining_text = re.sub(r'^[\:\-\ـ\.]+\s*', '', remaining_text)

            # 4. إعادة بناء السطر بشكل معياري وموحد
            if remaining_text:
                # إذا كان هناك نص في نفس السطر (مثل: "الهيئة العامة:" أو بداية الفقرة)
                clean_header = f"## المادة {article_num}:\n{remaining_text}"
            else:
                # إذا كانت المادة برقمها فقط
                clean_header = f"## المادة {article_num}:"

            normalized_lines.append(clean_header)
        else:
            # إذا لم يكن السطر يحتوي على مادة، نتركه كما هو
            normalized_lines.append(line)

    # إعادة دمج النص مع الحفاظ على الأسطر
    return '\n'.join(normalized_lines)

# ==========================================
# تجربة الكود على عينة من الحالات المشوهة
# ==========================================
sample_text = """
---
document_type: نظام
document_title: نظام صندوق الطلبة في جامعة الطفيلة التقنية
issuing_authority: ملك المملكة الأردنية الهاشمية
document_year: 2016
decision_number: ٢٠
legal_basis: المادة (٣٣) من قانون الجامعات الأردنية رقــم (٢٠) لسنة ٢٠٠٩
---
نحن عبدالله الثاني ابن الحسين ملك المملكة الاردنية الهاشمية بمقتضى المسادة (٣١) مسن الدست ور وبناء على ما قرره مجلس السوزراء بتاريخ ٢٠١٦/٣/٣٠ نأمر بوضع النظام الآتي :-
"""
cleaned_text = standardize_article_headers(sample_text)
print(cleaned_text)

# **1. Code Parse Tree v1**

In [ ]:
import re
import json
import yaml

# ==========================================
# Document Splitter
# ==========================================
class LegalDocumentSplitter:
    """
    طبقة وسيطة لتقطيع النظام أو التعليمات إلى مواد مستقلة (Chunks)
    قبل إرسالها لآلة الهيكلة.
    """
    def __init__(self):
        self.article_start_pattern = re.compile(r'^[\#\-\s]*المادة\s*[\(]?\s*(\d+|[٠-٩]+)\s*[\)]?', re.MULTILINE)

    def normalize_numbers(self, text):
        eastern_to_western = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
        return text.translate(eastern_to_western)

    def split_document(self, text):
        text = self.normalize_numbers(text)
        matches = list(self.article_start_pattern.finditer(text))

        chunks = []
        if not matches:
            return [{"chunk_type": "full_text", "text": text.strip()}]

        preamble = text[:matches[0].start()].strip()
        if preamble:
            chunks.append({
                "chunk_type": "preamble",
                "text": preamble
            })

        for i in range(len(matches)):
            start_idx = matches[i].start()
            end_idx = matches[i+1].start() if i + 1 < len(matches) else len(text)

            article_text = text[start_idx:end_idx].strip()
            article_num = matches[i].group(1)

            chunks.append({
                "chunk_type": "article",
                "article_number": article_num,
                "text": article_text
            })

        return chunks

class LegalTextStructurer:
    """
    أداة هيكلة النصوص القانونية (النسخة المحسنة ضد تشوهات OCR):
    - مرونة عالية مع فواصل الفروع والبنود (تعمل حتى لو نسي الـ OCR وضع شحطة).
    - معالجة ذكية للأسطر المبتورة داخل مادة التعريفات.
    - تدمير متقدم للتواريخ الوهمية والشوائب المخفية.
    """
    def __init__(self):
        # Regex Patterns
        self.article_pattern = re.compile(r'^[\#\-\s]*المادة\s*[\(]?\s*(\d+)\s*[\)]?\s*[\:\-\ـ]*\s*(.*)$')
        self.axis_pattern = re.compile(r'^[\#\-\s]*\**((?:أولاً|ثانياً|ثالثاً|رابعاً|خامساً))[\:\-\.\*]*\s*(.*)$')

        self.branch_pattern = re.compile(r'^[\#\-\s\*]*([أبتثجحخدذرزسشصضطظعغفقكلمنهوي])[\.\-\:\ـ\)]*\s+(.*)$')

        self.item_pattern = re.compile(r'^[\#\-\s\*]*(\d+)[\.\-\:\ـ\)]*\s+(.*)$')

        self.def_intro_pattern = re.compile(r'(يكون للكلمات|تكون للكلمات|معاني الكلمات|الكلمات والعبارات)')

    def extract_frontmatter(self, text):
        metadata = {}
        content = text
        match = re.match(r'^---\s*\n(.*?)\n---\s*\n(.*)', text, flags=re.DOTALL)
        if match:
            yaml_content = match.group(1)
            content = match.group(2)
            try:
                metadata = yaml.safe_load(yaml_content) or {}
            except yaml.YAMLError as e:
                print(f"❌ Error Douring Reading Metadata: {e}")
        return metadata, content

    def normalize_arabic_numbers(self, text):
        eastern_to_western = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
        return text.translate(eastern_to_western)

    def standardize_article_headers(self, text):
        text = self.normalize_arabic_numbers(text)
        text = re.sub(r'المادة\s+و\b', 'المادة 9', text)
        lines = text.split('\n')
        normalized_lines = []

        for line in lines:
            line = re.sub(r'[¹²³⁴⁵⁶⁷⁸⁹⁰]+$', '', line.strip())
            match = self.article_pattern.match(line)
            if match:
                article_num = match.group(1)
                remaining_text = match.group(2).strip()
                remaining_text = re.sub(r'^[\:\-\ـ\.]+\s*', '', remaining_text)
                if remaining_text:
                    clean_header = f"## المادة {article_num}:\n{remaining_text}"
                else:
                    clean_header = f"## المادة {article_num}:"
                normalized_lines.append(clean_header)
            else:
                normalized_lines.append(line)
        return '\n'.join(normalized_lines)

    def eradicate_noise(self, text):
        lines = text.split('\n')
        clean_lines = []

        for line in lines:
            line = line.replace(r'\_', '-')

            if line.count('|') >= 2: continue
            if re.search(r'[A-Za-z]{5,}', line): continue
            if "عبد الله الثاني" in line or "دائرة الرئاسة" in line: continue
            if re.search(r'صفحة\s*\d+\s*من\s*\d+', line) or re.match(r'^\d{4}/\d{1,2}/\d{1,2}', line.strip()): continue

            # OCR (مثل ## 7.17/٣/٣.)
            if re.match(r'^[\#\-\s]*\d+[\.\/\\]+\d+', line.strip()): continue

            if line.strip().startswith('!['): continue

            line = re.sub(r'[¹²³⁴⁵⁶⁷⁸⁹⁰]+', '', line)
            line = re.sub(r'\$_\d+\$', '', line)

            if line.strip(): clean_lines.append(line.strip())

        return clean_lines

    def merge_orphans(self, lines):
        merged = []
        i = 0
        while i < len(lines):
            line = lines[i]
            if re.match(r'^[\#\-\s\*]*([أ-ي]|\d+)[\.\-\:\ـ\)]*\s*$', line):
                if i + 1 < len(lines):
                    line = f"{line.strip()} {lines[i+1].strip()}"
                    i += 1
            merged.append(line)
            i += 1
        return merged


    def process_definitions(self, lines):
        intro = []
        definitions = {}
        last_term = None

        for line in lines:
            if self.def_intro_pattern.search(line) and not definitions:
                intro.append(line)
            else:
                clean_line = re.sub(r'_+', '', line).replace('**', '')
                parts = re.split(r'[:\-]', clean_line, maxsplit=1)

                if len(parts) == 2 and parts[0].strip() and parts[1].strip():
                    term = parts[0].strip()
                    definition = parts[1].strip()
                    definitions[term] = definition
                    last_term = term
                else:
                    if last_term:
                        definitions[last_term] += " " + clean_line
                    else:
                        intro.append(clean_line)

        return {"intro": " ".join(intro), "definitions": definitions}

    def build_tree(self, lines):
        tree = {
            "title": "",
            "intro": "",
            "axes": {},
            "branches": {},
            "items": []
        }
        current_axis = None
        current_branch = None

        for line in lines:
            if line.startswith("## المادة"):
                tree["title"] = line.replace("## ", "").strip()
                continue

            axis_match = self.axis_pattern.match(line)
            if axis_match:
                current_axis = axis_match.group(1)
                current_branch = None
                tree["axes"][current_axis] = {"content": axis_match.group(2), "branches": {}, "items": []}
                continue

            branch_match = self.branch_pattern.match(line)
            if branch_match:
                current_branch = branch_match.group(1)
                if current_branch == 'ه': current_branch = 'هـ'
                branch_node = {"content": branch_match.group(2), "items": []}

                if current_axis: tree["axes"][current_axis]["branches"][current_branch] = branch_node
                else: tree["branches"][current_branch] = branch_node
                continue

            item_match = self.item_pattern.match(line)
            if item_match:
                item_node = {"id": item_match.group(1), "content": item_match.group(2)}
                if current_branch and current_axis: tree["axes"][current_axis]["branches"][current_branch]["items"].append(item_node)
                elif current_branch: tree["branches"][current_branch]["items"].append(item_node)
                elif current_axis: tree["axes"][current_axis]["items"].append(item_node)
                else: tree["items"].append(item_node)
                continue

            if current_branch and current_axis: tree["axes"][current_axis]["branches"][current_branch]["content"] += " " + line
            elif current_branch: tree["branches"][current_branch]["content"] += " " + line
            elif current_axis: tree["axes"][current_axis]["content"] += " " + line
            elif not tree["branches"] and not tree["axes"] and not tree["items"]:
                tree["intro"] += (" " + line if tree["intro"] else line)

        return tree

    # Document Splitter
    def structure_single_article(self, raw_article_text):
        standardized_text = self.standardize_article_headers(raw_article_text)
        clean_lines = self.eradicate_noise(standardized_text)
        merged_lines = self.merge_orphans(clean_lines)

        if not merged_lines:
            return {}

        is_definitions = any(self.def_intro_pattern.search(line) for line in merged_lines[:3])

        if is_definitions:
            article_data = self.process_definitions(merged_lines)
            article_data["type"] = "definitions"
            for line in merged_lines:
                if line.startswith("## المادة"):
                    article_data["title"] = line.replace("## ", "").strip()
                    break
        else:
            article_data = self.build_tree(merged_lines)
            article_data["type"] = "hierarchical"

        return article_data

    # ==========================================
    # Orchestrator
    # ==========================================
    def structure_article(self, raw_text):
        metadata, core_text = self.extract_frontmatter(raw_text)

        standardized_text = self.standardize_article_headers(core_text)
        clean_lines = self.eradicate_noise(standardized_text)
        merged_lines = self.merge_orphans(clean_lines)

        if not merged_lines:
            return {"document_metadata": metadata, "article_data": {}}

        is_definitions = any(self.def_intro_pattern.search(line) for line in merged_lines[:3])

        if is_definitions:
            article_data = self.process_definitions(merged_lines)
            article_data["type"] = "definitions"
            for line in merged_lines:
                if line.startswith("## المادة"):
                    article_data["title"] = line.replace("## ", "").strip()
                    break
        else:
            article_data = self.build_tree(merged_lines)
            article_data["type"] = "hierarchical"

        final_structure = {
            "document_metadata": metadata,
            "article_data": article_data
        }

        return final_structure

# ==========================================
# The Grand Orchestrator
# ==========================================
def process_full_legal_document(raw_markdown):
    """
    الـ Pipeline الكامل: Metadata -> Split -> Structure -> JSON
    """
    # Extract Metadata
    metadata = {}
    content = raw_markdown
    match = re.match(r'^---\s*\n(.*?)\n---\s*\n(.*)', raw_markdown, flags=re.DOTALL)

    if match:
        try:
            metadata = yaml.safe_load(match.group(1)) or {}
            content = match.group(2)
        except:
            pass

    splitter = LegalDocumentSplitter()
    structurer = LegalTextStructurer()

    chunks = splitter.split_document(content)

    final_output = {
        "document_metadata": metadata,
        "preamble": "",
        "articles": []
    }

    for chunk in chunks:
        if chunk["chunk_type"] == "preamble":
            final_output["preamble"] = chunk["text"]
        elif chunk["chunk_type"] == "article":
            structured_article=structurer.structure_article(chunk["text"])
            #structured_article = structurer.structure_single_article(chunk["text"])
            structured_article["article_number"] = chunk["article_number"]
            final_output["articles"].append(structured_article)

    return final_output

# ==========================================
# Test Run
# ==========================================
# if __name__ == "__main__":
#     sample_raw_text = """ """
#     result = process_full_legal_document(sample_raw_text)
#     print(json.dumps(result, ensure_ascii=False, indent=2))

# **Preprocess all file in the Drive**

In [ ]:
import os
import json
from pathlib import Path

def process_and_export_drive_documents(input_base_dir, output_base_dir):
    """
    محرك أتمتة يقرأ جميع ملفات MD من مجلدات Drive، يعالجها بالكود المخصص،
    ويصدرها كملفات JSON مهيكلة مع الحفاظ على شجرة المجلدات.
    """
    input_path = Path(input_base_dir)
    output_path = Path(output_base_dir)

    output_path.mkdir(parents=True, exist_ok=True)

    md_files = list(input_path.rglob("*.md"))

    if not md_files:
        print("Path not Found")
        return

    print(f"🔍 Files Found {len(md_files)} ")
    print("-" * 40)

    success_count = 0
    error_count = 0

    for file_path in md_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_content = f.read()

            # The Grand Orchestrator
            structured_data = process_full_legal_document(raw_content)

            folder_category = file_path.parent.name
            if 'document_metadata' in structured_data:
                structured_data['document_metadata']['source_folder'] = folder_category
                structured_data['document_metadata']['original_filename'] = file_path.name

            relative_path = file_path.relative_to(input_path)
            output_file_path = output_path / relative_path.with_suffix('.json')

            output_file_path.parent.mkdir(parents=True, exist_ok=True)

            with open(output_file_path, 'w', encoding='utf-8') as f:
                json.dump(structured_data, f, ensure_ascii=False, indent=4)

            success_count += 1
            print(f"✅ Process Done: {file_path.name}")

        except Exception as e:
            print(f"❌ Error Douring Proces File{file_path.name}: {e}")
            error_count += 1

    print("-" * 40)
    print(f"🎯 success: {success_count} | error: {error_count}")
    print(f"📂 : {output_path}")

# ==========================================
# Execution Zone
# ==========================================
if __name__ == "__main__":
    import os


    # if not os.path.exists('/content/drive/MyDrive'):
    #     from google.colab import drive
    #     drive.mount('/content/drive')
    # else:
    #     print("✅ Google Drive مربوط مسبقاً، جاري بدء المعالجة مباشرة...")

    INPUT_DIRECTORY = '/content/drive/MyDrive/low_TTU_md'

    OUTPUT_DIRECTORY = '/content/drive/MyDrive/legal_TTU_JSON'

    process_and_export_drive_documents(INPUT_DIRECTORY, OUTPUT_DIRECTORY)

# **كود تهيئة النموذج**

In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-chroma langchain-huggingface

In [ ]:
!pip install -U \
langchain \
langchain-community \
langchain-chroma \
langchain-huggingface \
chromadb \
sentence-transformers \
transformers \
accelerate

In [ ]:
!pip list | grep langchain

In [ ]:
import os
os.kill(os.getpid(), 9)

# **It reads JSON files from the Drive, converts them to a Parent-Child structure, stores them in the DB, and then uploads them to the Drive.**

In [ ]:
# =========================================================
# Imports
# =========================================================

import json
from pathlib import Path

from langchain_core.documents import Document

from langchain_chroma import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings


In [ ]:
# =========================================================
# Embedding Model
# =========================================================

EMBED_MODEL = "intfloat/multilingual-e5-large"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("✅ Embedding model loaded")

In [ ]:
# =========================================================
# Paths
# =========================================================

JSON_DIR = Path("/content/drive/MyDrive/legal_TTU_JSON")

CHROMA_DIR = "/content/chroma_TTU_legal_db"

# =========================================================
# Text Splitter
# =========================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=[
        "\n\n",
        "\n",
        ".",
        " "
    ]
)

# =========================================================
# Vector DB
# =========================================================

vectorstore = Chroma(
    collection_name="ttu_legal_collection",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

# =========================================================
# Load JSON Files
# =========================================================

json_files = list(JSON_DIR.rglob("*.json"))

print(f"📂 Found {len(json_files)} JSON files")

all_chunks = []

# =========================================================
# Process Files
# =========================================================

for file_path in json_files:

    try:

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        metadata = data.get("document_metadata", {})

        doc_title = metadata.get(
            "document_title",
            "Unknown Document"
        )

        doc_type = metadata.get(
            "document_type",
            "Unknown Type"
        )

        # ================================================
        # Remove metadata from content
        # ================================================

        content_only = {
            k: v
            for k, v in data.items()
            if k != "document_metadata"
        }

        parent_text = (
            f"passage: {doc_type} - {doc_title}\n\n"
            + json.dumps(
                content_only,
                ensure_ascii=False,
                indent=2
            )
        )

        # ================================================
        # Split into child chunks
        # ================================================

        chunks = splitter.split_text(parent_text)

        # ================================================
        # Convert to Documents
        # ================================================

        for idx, chunk in enumerate(chunks):

            chunk_doc = Document(
                page_content=chunk,
                metadata={
                    "source": str(file_path),
                    "document_title": doc_title,
                    "document_type": doc_type,
                    "chunk_id": idx
                }
            )

            all_chunks.append(chunk_doc)

    except Exception as e:

        print(f"❌ Error in file: {file_path}")
        print(e)

# =========================================================
# Add to Chroma
# =========================================================

print(f"🧠 Indexing {len(all_chunks)} chunks...")

vectorstore.add_documents(all_chunks)

print("✅ Database created successfully")

# =========================================================
# Create Retriever
# =========================================================

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

print("🚀 Retriever ready")

# **run Leagel RAG**

In [ ]:
!pip install -U langchain-groq

In [ ]:
# =========================================================
# 1. Imports
# =========================================================

import os

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser

In [ ]:
# =========================================================
# 2. API KEY
# =========================================================

os.environ["GROQ_API_KEY"] = "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"

# =========================================================
# 4. Load Chroma DB from Drive
# =========================================================

CHROMA_DIR = "/content/drive/MyDrive/chroma_TTU_legal_db"

vectorstore = Chroma(
    collection_name="ttu_legal_collection",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

print("✅ Chroma DB loaded")

In [ ]:
# =========================================================
# 5. Create Retriever
# =========================================================

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

print("✅ Retriever ready")

In [ ]:
# =========================================================
# 6. Load Groq Model
# =========================================================

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

print("✅ Groq model loaded")

In [ ]:
# =========================================================
# 7. Legal Prompt
# =========================================================

LEGAL_PROMPT = """
أنت مستشار أكاديمي وقانوني ذكي ومتخصص في أنظمة وتعليمات جامعة الطفيلة التقنية،
وخاصة القضايا المتعلقة بشؤون الطلبة.

دورك ليس فقط نقل النصوص، بل:
- فهم سؤال الطالب
- تحليل التعليمات الجامعية
- شرح الأنظمة بطريقة واضحة وبسيطة
- تقديم استشارة أكاديمية دقيقة ومهنية
- استنتاج الإجابة الأقرب من السياق عند عدم وجود نص مباشر

التزم بالقواعد التالية:

1- اعتمد بشكل أساسي على السياق المسترجع من وثائق الجامعة.
2- يمكنك الاستنتاج بشكل محدود ومنطقي إذا كانت المعلومات غير مباشرة، بشرط أن يكون الاستنتاج مبنياً على السياق المتوفر.
3- لا تخترع مواد أو قرارات أو تعليمات غير موجودة.
4- إذا لم تجد إجابة مباشرة، حاول:
   - تحليل المواد المرتبطة
   - شرح الاحتمالات
   - تقديم أقرب تفسير منطقي
   بدلاً من إنهاء الإجابة مباشرة بعبارة عدم توفر المعلومات.
5- استخدم أسلوب استشاري احترافي وكأنك موظف خبير في شؤون الطلبة.
6- اجعل الإجابة سهلة الفهم للطالب وغير جامدة.
7- إذا كانت هناك شروط أو استثناءات مهمة فاذكرها بوضوح.
8- إذا وُجد تعارض أو اختلاف بين التعليمات فقم بتوضيح ذلك.
9- إذا كانت المعلومات غير مؤكدة بشكل كامل، وضّح ذلك بصراحة.
10- في نهاية الإجابة اذكر التعليمات أو الوثائق التي استندت إليها.

أسلوب الإجابة:
- واضح
- منظم
- مرن
- احترافي
- مختصر عند الحاجة
- تفصيلي عند الحاجة
- غير آلي أو جامد

All responses must be generated strictly in valid Markdown format.
Use clear and well-structured Markdown styling, including headings, subheadings, bullet lists, numbered lists, emphasis, tables, and fenced code blocks whenever appropriate.

Ensure that:
- Markdown syntax is always valid and properly closed.
- Responses remain visually organized and easy to read.
- Code, JSON, commands, and configuration snippets are always enclosed in fenced code blocks.
- Plain unformatted text is avoided unless explicitly required.

السؤال:
{question}

السياق المسترجع:
{context}

الإجابة:
"""

prompt = ChatPromptTemplate.from_template(
    LEGAL_PROMPT
)

In [ ]:
# =========================================================
# 8. Helper Function
# =========================================================

def format_docs(docs):

    formatted = []

    for i, doc in enumerate(docs, start=1):

        source = doc.metadata.get(
            "document_title",
            "Unknown Source"
        )

        chunk = f"""
[المصدر {i}]
اسم الوثيقة: {source}

النص:
{doc.page_content}
"""

        formatted.append(chunk)

    return "\n\n".join(formatted)

In [ ]:
# =========================================================
# 9. Ask Function
# =========================================================

def ask_legal_rag(question):

    # =============================================
    # E5 Query Formatting
    # =============================================

    formatted_query = f"query: {question}"

    # =============================================
    # Retrieve Documents
    # =============================================

    docs = retriever.invoke(formatted_query)

    # =============================================
    # Build Context
    # =============================================

    context = format_docs(docs)

    # =============================================
    # Chain
    # =============================================

    chain = (
        prompt
        | llm
        | StrOutputParser()
    )

    # =============================================
    # Generate Answer
    # =============================================

    response = chain.invoke({
        "question": question,
        "context": context
    })

    # =============================================
    # Print Sources
    # =============================================
    if docs:
        sources_md = "\n\n---\n### 📚 المصادر المرجعية:\n"
        seen_sources = set()
        counter = 1

        for doc in docs:
            # Fallback to default if not found
            title = doc.metadata.get("document_title", "وثيقة غير مسماة")
            doc_type = doc.metadata.get("document_type", "غير محدد")

            source_identifier = f"{title}-{doc_type}"

            if source_identifier not in seen_sources:
                seen_sources.add(source_identifier)
                sources_md += f"{counter}. **{title}** ({doc_type})\n"
                counter += 1

        final_output = response + sources_md
    else:
        final_output = response + "\n\n---\n*لم يتم العثور على مصادر مطابقة في قاعدة البيانات.*"

    return final_output

# =========================================================
# 10. Test
# =========================================================

question = "ما هي حقوق الطالب في جامعة الطفيلة التقنية"

answer = ask_legal_rag(question)

print("\n==============================")
print(" الإجابة:")
print("==============================\n")

print(answer)

In [ ]:
import gradio as gr

In [ ]:
def clean_query(massag , hestory):
    try:
        if isinstance(massag , dict):
            answer = massag.get("text" , "")
        else:
            answer = str(massag)

        if answer == "":
            return "الرجاء كتابة سؤال"

        return ask_legal_rag(answer)

    except Exception as e:
        return f"حدث خطأ :{type(e)}:{str(e)}"


In [ ]:
import gradio as gr
css = """
/* توجيه الحاوية الرئيسية بالكامل */
.gradio-container {
    direction: rtl !important;
    font-family: 'Tajawal', 'Arial', sans-serif;
}

/* --------------------------------------------------- */
/* الإصلاح الجذري لنصوص الماركدوان داخل الشات بوت (Prose) */
/* --------------------------------------------------- */
.prose {
    direction: rtl !important;
    text-align: right !important;
}

/* توجيه الفقرات والنصوص العادية */
.prose p, .prose span, .prose strong, .prose em {
    direction: rtl !important;
    text-align: right !important;
}

/* إصلاح مشكلة القوائم النقطية والرقمية */
.prose ul, .prose ol {
    direction: rtl !important;
    text-align: right !important;
    padding-right: 25px !important; /* نقل المسافة البادئة لليمين */
    padding-left: 0 !important;     /* إلغاء المسافة البادئة من اليسار */
}

/* توجيه عناصر القائمة نفسها */
.prose li {
    direction: rtl !important;
    text-align: right !important;
    margin-right: 0 !important;
}

/* --------------------------------------------------- */
/* توجيه باقي عناصر الواجهة */
/* --------------------------------------------------- */
.message {
    direction: rtl !important;
    text-align: right !important;
    font-size: 18px !important;
}

textarea {
    direction: rtl !important;
    text-align: right !important;
    font-size: 16px !important;
}

.bot, .user {
    text-align: right !important;
}
"""


description_text = """$يحوي 2 نظام و11 تعليمات، خاصة بشؤون الطلبة التي في الموقع
https://eportal.ttu.edu.jo/regulation_view/

الأنظمة:
- نظام تأديب الطلبة في جامعة الطفيلة التقنية
- نظام صندوق الطلبة في جامعة الطفيلة التقنية

التعليمات:
- أسس العمل التطوعي وخدمة المجتمع في جامعة الطفيلة التقنية
- التعليمات التنفيذية لنظام تأديب الطلبة في جامعة الطفيلة التقنية
- التعليمات التنفيذية لنظام صندوق الطلبة في جامعة الطفيلة التقنية
- تعليمات إصدار الهوية الجامعية لطلبة جامعة الطفيلة التقنية
- تعليمات الأندية الطلابية في جامعة الطفيلة التقنية
- تعليمات التأمين الصحي للطلبة في الجامعة
- تعليمات التدريب الميداني لطلبة البكالوريوس
- تعليمات التدريب الميداني لطلبة البكالوريوس المقبولين قبل العام الجامعي 2016-2017
- تعليمات الدراسة الخاصة الحرة في جامعة الطفيلة التقنية
- تعليمات الرحلات الطلابية للطلبة في جامعة الطفيلة التقنية
- تعليمات السكن الداخلي للطالبات في جامعة الطفيلة التقنية
- تعليمات النشاط الكشفي في جامعة الطفيلة التقنية
- تعليمات مادة مشروع البحث لطلبة البكالوريوس في جامعة الطفيلة التقنية
- تعليمات مشروع التخرج لطلبة كلية الهندسة$
"""

demo = gr.ChatInterface(
    fn=clean_query,
    title="مساعد قانوني مختص في القوانين الخاصة بشؤون الطلبة",
    description=description_text,
    css=css,
    theme=gr.themes.Soft(),
    textbox=gr.Textbox(placeholder="اكتب استفسارك القانوني هنا...", container=False, scale=7)
)

if __name__ == "__main__":
    demo.launch()